# Fraud detection pipeline — Bronze / Silver / Gold

Ingesting the Kaggle credit card fraud dataset (creditcard.csv), build it through
the medallion architecture, and then landing a Gold table ready for dashboards and modelling.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "main"
SCHEMA = "fraud_project"
RAW_PATH = "/Volumes/main/fraud_project/raw_data/creditcard.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

## Bronze — raw ingestion
Load the file exactly as-is. No cleaning, no filtering. This is my
untouched copy for traceability and reprocessing.

In [0]:
bronze_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(RAW_PATH)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    bronze_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_transactions")
)

print(f"Bronze row count: {bronze_df.count()}")
display(bronze_df.limit(5))

Bronze row count: 284807


Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,_ingested_at,_source_file
0.0,-1.3598071336738,-0.0727811733098497,2.53634673796914,1.37815522427443,-0.338320769942518,0.462387777762292,0.239598554061257,0.0986979012610507,0.363786969611213,0.0907941719789316,-0.551599533260813,-0.617800855762348,-0.991389847235408,-0.311169353699879,1.46817697209427,-0.470400525259478,0.207971241929242,0.0257905801985591,0.403992960255733,0.251412098239705,-0.018306777944153,0.277837575558899,-0.110473910188767,0.0669280749146731,0.128539358273528,-0.189114843888824,0.133558376740387,-0.0210530534538215,149.62,0,2026-10-08T00:46:59.033Z,dbfs:/Volumes/main/fraud_project/raw_data/creditcard.csv
0.0,1.19185711131486,0.26615071205963,0.16648011335321,0.448154078460911,0.0600176492822243,-0.0823608088155687,-0.0788029833323113,0.0851016549148104,-0.255425128109186,-0.166974414004614,1.61272666105479,1.06523531137287,0.48909501589608,-0.143772296441519,0.635558093258208,0.463917041022171,-0.114804663102346,-0.183361270123994,-0.145783041325259,-0.0690831352230203,-0.225775248033138,-0.638671952771851,0.101288021253234,-0.339846475529127,0.167170404418143,0.125894532368176,-0.00898309914322813,0.0147241691924927,2.69,0,2026-10-08T00:46:59.033Z,dbfs:/Volumes/main/fraud_project/raw_data/creditcard.csv
1.0,-1.35835406159823,-1.34016307473609,1.77320934263119,0.379779593034328,-0.503198133318193,1.80049938079263,0.791460956450422,0.247675786588991,-1.51465432260583,0.207642865216696,0.624501459424895,0.066083685268831,0.717292731410831,-0.165945922763554,2.34586494901581,-2.89008319444231,1.10996937869599,-0.121359313195888,-2.26185709530414,0.524979725224404,0.247998153469754,0.771679401917229,0.909412262347719,-0.689280956490685,-0.327641833735251,-0.139096571514147,-0.0553527940384261,-0.0597518405929204,378.66,0,2026-10-08T00:46:59.033Z,dbfs:/Volumes/main/fraud_project/raw_data/creditcard.csv
1.0,-0.966271711572087,-0.185226008082898,1.79299333957872,-0.863291275036453,-0.0103088796030823,1.24720316752486,0.23760893977178,0.377435874652262,-1.38702406270197,-0.0549519224713749,-0.226487263835401,0.178228225877303,0.507756869957169,-0.28792374549456,-0.631418117709045,-1.0596472454325,-0.684092786345479,1.96577500349538,-1.2326219700892,-0.208037781160366,-0.108300452035545,0.00527359678253453,-0.190320518742841,-1.17557533186321,0.647376034602038,-0.221928844458407,0.0627228487293033,0.0614576285006353,123.5,0,2026-10-08T00:46:59.033Z,dbfs:/Volumes/main/fraud_project/raw_data/creditcard.csv
2.0,-1.15823309349523,0.877736754848451,1.548717846511,0.403033933955121,-0.407193377311653,0.0959214624684256,0.592940745385545,-0.270532677192282,0.817739308235294,0.753074431976354,-0.822842877946363,0.53819555014995,1.3458515932154,-1.11966983471731,0.175121130008994,-0.451449182813529,-0.237033239362776,-0.0381947870352842,0.803486924960175,0.408542360392758,-0.00943069713232919,0.79827849458971,-0.137458079619063,0.141266983824769,-0.206009587619756,0.502292224181569,0.219422229513348,0.215153147499206,69.99,0,2026-10-08T00:46:59.033Z,dbfs:/Volumes/main/fraud_project/raw_data/creditcard.csv


## Silver — clean and conform
Deduplicate, enforce types, handle nulls, and run basic data quality checks.
The Kaggle dataset is already fairly clean, so this mostly demonstrates the
pattern you'd apply to messier real-world data.

In [0]:
bronze_df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_transactions")

# Data quality checks — fail loudly rather than silently dropping rows
null_counts = bronze_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["Time", "Amount", "Class"]]
).collect()[0].asDict()

for col_name, n_nulls in null_counts.items():
    if n_nulls > 0:
        print(f"WARNING: {n_nulls} nulls found in {col_name}")

In [0]:
silver_df = (
    bronze_df
    .dropDuplicates([c for c in bronze_df.columns if not c.startswith("_")])
    .withColumn("Amount", F.col("Amount").cast("double"))
    .withColumn("Class", F.col("Class").cast("int"))
    .withColumn("transaction_hour", (F.col("Time") % 86400 / 3600).cast("int"))
    .filter(F.col("Amount") >= 0)  # negative amounts would be a data error here
    .withColumnRenamed("Class", "is_fraud")
    .drop("_source_file")
)

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_transactions")
)

print(f"Silver row count: {silver_df.count()}")
print(f"Fraud rate: {silver_df.filter(F.col('is_fraud') == 1).count() / silver_df.count():.4%}")

Silver row count: 283726
Fraud rate: 0.1667%


## Gold — business aggregations
Two Gold tables: an hourly fraud-rate rollup for the dashboard, and a
feature table for model training (built out further in the next notebook).

In [0]:
silver_df = spark.table(f"{CATALOG}.{SCHEMA}.silver_transactions")

gold_hourly_summary = (
    silver_df.groupBy("transaction_hour")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("is_fraud").alias("fraud_count"),
        F.round(F.avg("Amount"), 2).alias("avg_amount"),
        F.round(F.sum("is_fraud") / F.count("*") * 100, 3).alias("fraud_rate_pct"),
    )
    .orderBy("transaction_hour")
)

(
    gold_hourly_summary.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_hourly_fraud_summary")
)

display(gold_hourly_summary)

transaction_hour,transaction_count,fraud_count,avg_amount,fraud_rate_pct
0,7647,6,60.59,0.078
1,4208,10,62.77,0.238
2,3308,48,70.41,1.451
3,3487,17,51.76,0.488
4,2204,23,77.16,1.044
5,2988,11,50.7,0.368
6,4082,9,65.19,0.22
7,7233,23,67.89,0.318
8,10232,9,88.89,0.088
9,15767,16,102.93,0.101
